# Huấn luyện mô hình bất nhất cục bộ
Bật **GPU + Internet**, điền REPO_URL của nhóm ở ô dưới rồi chạy từ trên xuống. Notebook clone code từ GitHub; không cần upload gói nguồn.
CNN SyncNet được giữ cố định. Notebook fine-tune lớp chiếu pretrained và huấn luyện mạng thời gian, tạo local.pt mới.


In [ ]:
from pathlib import Path
import sys
import subprocess
import json
import shutil

REPO_URL = "https://github.com/YOUR_USERNAME/vn-av-forensics.git"
REPO_REF = "main"  # Có thể thay bằng commit SHA để tái lập một phiên bản.
ROOT = Path("/kaggle/working/vn-av-forensics")

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Điền URL repository GitHub của nhóm vào REPO_URL")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
if not (ROOT / ".git").is_dir():
    raise ValueError("ROOT đã tồn tại nhưng không phải Git repo; chọn folder ROOT mới")
def git(*args):
    return subprocess.check_output(["git", "-C", str(ROOT), *args], text=True).strip()
if git("remote", "get-url", "origin") != REPO_URL:
    raise ValueError("Folder ROOT đang chứa repository khác; chọn ROOT mới")
if git("status", "--porcelain"):
    raise ValueError("Checkout có thay đổi chưa lưu; commit chúng hoặc chọn ROOT mới")
subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", REPO_REF], check=True)
target_commit = git("rev-parse", "FETCH_HEAD")
if target_commit != git("rev-parse", "HEAD") and "avtc" in sys.modules:
    raise RuntimeError("Code đã cập nhật: restart kernel trước khi chạy lại notebook")
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", target_commit], check=True)
CODE_VERSION = {"repository": REPO_URL, "requested_ref": REPO_REF, "commit": target_commit}
print("Code version:", CODE_VERSION)

# Install dependencies in the cloned project.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               cwd=ROOT, check=True)
sys.path.insert(0, str(ROOT / "src"))
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Bật GPU trước khi chạy")
print(torch.cuda.get_device_name(0))


## Dữ liệu
Mặc định: 6 người GRID, 40 video/người; 4 người train, 1 validation, 1 test. Đây là cấu hình khởi đầu, cần mở rộng người nói trước khi kết luận tổng quát.
Chỉ tạo lệch cục bộ với vị trí và mức lệch thay đổi giữa video. Original và biến đổi đồng thời cả hai luồng là mẫu âm/đối chứng. Độ lệch chung của nguồn chỉ được bù trong tiền xử lý.
Nếu đã có dữ liệu, đặt MANIFEST thành đường dẫn JSON gồm id, speaker, video, audio (tùy chọn), split (train/validation/test). Không chia các biến thể cùng nguồn sang các tập khác nhau.


In [ ]:
from avtc.data import download_grid, prepare_dataset
from avtc.syncnet import setup
ASSETS=Path("/kaggle/working/checkpoints/syncnet")
setup(ASSETS)
MANIFEST=None
if MANIFEST is None:
    MANIFEST=download_grid("/tmp/grid",speakers=(1,2,3,4,5,6),clips=40)
CACHE=prepare_dataset(MANIFEST,"/tmp/prepared",ASSETS,device="cuda",seed=42)
index=json.loads((CACHE/"index.json").read_text())
import pandas as pd
display(pd.DataFrame(index["rows"]).groupby(["split","variant"]).size().unstack())
print((CACHE/"excluded.json").read_text())


## Kiểm tra đầu vào
Xem một crop gốc và nghe tiếng trước khi tiếp tục. GRID được giả định đồng bộ, phép bù bằng model chưa thay thế được kiểm tra thủ công. Nếu sai người/câu hoặc lệch bất thường, sửa nguồn và chuẩn bị lại ở thư mục mới.
Time-warp giữ thứ tự nội dung nhưng có thể làm đổi pitch ở vùng chuyển tiếp; phải xem tỷ lệ báo nhầm trên synchronous-control trong kết quả.


In [ ]:
from avtc.syncnet import ffmpeg_env, run
from IPython.display import Video, display
record=next((CACHE/"sources").glob("*/preprocess/track.json"))
crop=json.loads(record.read_text())["crop"]
ff,env=ffmpeg_env(ASSETS)
preview=Path("/tmp/preview.mp4")
run([ff,"-y","-v","error","-i",crop,"-c:v","libx264","-pix_fmt","yuv420p","-c:a","aac",preview],env)
display(Video(str(preview),embed=True))


## Huấn luyện và fine-tuning
Mặc định FINE_TUNE=True: cập nhật các lớp Linear pretrained ở hai nhánh và mạng thời gian mới; CNN và thống kê BatchNorm cố định. Có thể đặt False để so sánh với chỉ học mạng thời gian. Nếu đổi chế độ, dùng thư mục TRAIN khác.
Model chọn theo validation loss, ngưỡng theo validation F1. Tập test chỉ được dùng trong ô đánh giá cuối. last.pt cho phép tiếp tục epoch khi còn cùng cache/cấu hình; không dùng checkpoint thử nghiệm nhỏ làm model chính thức.


In [ ]:
from avtc.train import fit
EPOCHS=10
FINE_TUNE=True
TRAIN=Path("/kaggle/working/training")
TRAIN.mkdir(parents=True, exist_ok=True)
version_file = TRAIN / "code_version.json"
if version_file.exists() and json.loads(version_file.read_text())["commit"] != CODE_VERSION["commit"]:
    raise ValueError("Code đổi phiên bản: dùng TRAIN mới để không trộn checkpoint cũ")
version_file.write_text(json.dumps(CODE_VERSION, indent=2), encoding="utf-8")
checkpoint=fit(CACHE,TRAIN,epochs=EPOCHS,fine_tune=FINE_TUNE,
               resume=(TRAIN/"last.pt").exists(),device="cuda",seed=42)
curves=pd.read_csv(TRAIN/"learning_curve.csv")
display(curves)
curves.set_index("epoch")[["train_loss","validation_loss"]].plot(title="Learning curves")


## Đánh giá trên người nói chưa dùng để train/chọn model
Đọc recall/IoU cùng tỷ lệ báo nhầm của original và synchronous-control. Nếu cả hai luồng cùng biến đổi mà vẫn báo nhiều, model có thể đang dùng dấu vết resampling. Kết quả GRID không chứng minh hiệu quả deepfake hay tiếng Việt.


In [ ]:
from avtc.evaluate import evaluate
RESULTS=Path("/kaggle/working/results")
display(evaluate(CACHE,checkpoint,RESULTS,device="cuda"))
print((RESULTS/"metrics.json").read_text())
from IPython.display import Image
for plot in sorted(RESULTS.glob("*.png"))[:2]:display(Image(filename=str(plot)))


## Tải model và chạy demo
Sau Save Version/Run All, tải **training/local.pt** từ Kaggle Output, chép thành **checkpoints/local.pt** trên máy. Model tốt nhất và ngưỡng nằm trong cùng file.
Báo cáo Markdown, CSV và biểu đồ nằm trong **results/**; lịch sử train và last.pt để resume nằm trong **training/**. Không cần đóng gói.
Lần đầu chạy demo cần Internet để tải backbone/detector nếu checkpoints/syncnet chưa có.
Anaconda Prompt tại folder dự án:
```
conda create -n vnav python=3.11 -y
conda activate vnav
pip install -r requirements.txt
streamlit run app.py
```


In [ ]:
from avtc.syncnet import write_json
# Preserve lightweight data provenance; the heavy cache remains in /tmp.
DATA_INFO = RESULTS / "data"
DATA_INFO.mkdir(exist_ok=True)
for name in ["index.json", "prepare_config.json", "excluded.json"]:
    shutil.copy2(CACHE / name, DATA_INFO / name)
source_info = {}
for path in (CACHE / "sources").glob("*/source.json"):
    source_info[path.parent.name] = json.loads(path.read_text())
write_json(DATA_INFO / "sources.json", source_info)
print("Model cho demo:", checkpoint)
print("Resume checkpoint:", TRAIN / "last.pt")
print("Báo cáo:", RESULTS / "KET_QUA.md")
print("Lịch sử train:", TRAIN / "learning_curve.csv")
print("Git commit đã dùng:", CODE_VERSION["commit"])
